In [1]:
import csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
dataset = '../model/keypoint_classifier/keypoint.csv'
model_save_path = '../model/keypoint_classifier/keypoint_classifier.hdf5'
tflite_save_path = '../model/keypoint_classifier/keypoint_classifier.tflite'
NUM_CLASSES = 4

In [3]:
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=42)

In [5]:
model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(21 * 2, )),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout (Dropout)           (None, 42)                0         
                                                                 
 dense (Dense)               (None, 20)                860       
                                                                 
 dropout_1 (Dropout)         (None, 20)                0         
                                                                 
 dense_1 (Dense)             (None, 10)                210       
                                                                 
 dense_2 (Dense)             (None, 4)                 44        
                                                                 
Total params: 1114 (4.35 KB)
Trainable params: 1114 (4.35 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Early stopping callback
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [7]:
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
# Train Model
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000


22/24 [==========================>...] - ETA: 0s - loss: 1.3986 - accuracy: 0.2844 
Epoch 1: saving model to ../model/keypoint_classifier\keypoint_classifier.hdf5
24/24 [==============================] - 2s 19ms/step - loss: 1.3987 - accuracy: 0.2843 - val_loss: 1.3277 - val_accuracy: 0.4220
Epoch 2/1000
21/24 [=========================>....] - ETA: 0s - loss: 1.3362 - accuracy: 0.3378
Epoch 2: saving model to ../model/keypoint_classifier\keypoint_classifier.hdf5
24/24 [==============================] - 0s 6ms/step - loss: 1.3308 - accuracy: 0.3450 - val_loss: 1.2471 - val_accuracy: 0.4840


C:\Users\IshanDilhan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 3/1000
21/24 [=========================>....] - ETA: 0s - loss: 1.2670 - accuracy: 0.3966
Epoch 3: saving model to ../model/keypoint_classifier\keypoint_classifier.hdf5
24/24 [==============================] - 0s 6ms/step - loss: 1.2660 - accuracy: 0.3990 - val_loss: 1.1205 - val_accuracy: 0.6180
Epoch 4/1000
23/24 [===========================>..] - ETA: 0s - loss: 1.1928 - accuracy: 0.4555
Epoch 4: saving model to ../model/keypoint_classifier\keypoint_classifier.hdf5
24/24 [==============================] - 0s 6ms/step - loss: 1.1918 - accuracy: 0.4557 - val_loss: 1.0190 - val_accuracy: 0.7980
Epoch 5/1000
23/24 [===========================>..] - ETA: 0s - loss: 1.1318 - accuracy: 0.4973
Epoch 5: saving model to ../model/keypoint_classifier\keypoint_classifier.hdf5
24/24 [==============================] - 0s 6ms/step - loss: 1.1325 - accuracy: 0.4970 - val_loss: 0.9285 - val_accuracy: 0.8230
Epoch 6/1000
23/24 [===========================>..] - ETA: 0s - loss: 1.0678 - accuracy:

In [9]:
# Evaluate Model
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

8/8 [==============================] - 0s 3ms/step - loss: 0.2705 - accuracy: 0.9290


In [10]:
# Save as a model dedicated to inference
model.save(model_save_path, include_optimizer=False)

In [11]:
# Transform model (quantization) to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_quantized_model)
print("TFLite model saved!")

INFO:tensorflow:Assets written to: C:\Users\ISHAND~1\AppData\Local\Temp\tmpq9ay8kcu\assets


INFO:tensorflow:Assets written to: C:\Users\ISHAND~1\AppData\Local\Temp\tmpq9ay8kcu\assets


TFLite model saved!
